# 환자(Patients) 및 예약(Appointments) 데이터 정제 파이프라인
---
"""
환자(patients), 예약(appointments), 예약이벤트(appointment_events) 데이터 정제 및 이상치 격리 로직
- 중복 레코드(Exact Duplicates) 제거 및 Excluded 처리
- 필수 외래키 결측(blank patient_id) 및 시간 역전 이상치 격리(Quarantine)
- 한글 비표준 코드(남/여/미상, 네이버/카카오/전화) 표준 코드 매핑
- 최종 정제 완료(Clean) 데이터셋 산출 및 검증
"""

In [1]:
"""
환자 및 예약 정제 라이브러리 로드
"""
import pandas as pd
import numpy as np
import os
print("Environment initialized for Data Cleaning.")

In [2]:
"""
환자(Patients) 데이터 정제 로직
- sex_code: 남->M, 여->F, 미상->U 표준화
- active_yn: yes/no -> Y/N 표준화
- UNREGISTERED 미등록 환자 격리(Quarantine)
"""
def clean_patients(raw_df):
    df = raw_df.copy()
    sex_map = {"남": "M", "여": "F", "미상": "U", "M": "M", "F": "F", "U": "U"}
    df["sex_code"] = df["sex_code"].map(sex_map).fillna("U")
    df["active_yn"] = df["active_yn"].astype(str).str.upper().map({"YES": "Y", "NO": "N", "Y": "Y", "N": "N"}).fillna("N")
    quarantine_mask = df["patient_id"].str.contains("UNREG", na=False)
    clean_df = df[~quarantine_mask]
    quarantine_df = df[quarantine_mask]
    return clean_df, quarantine_df

print("clean_patients function defined.")

In [3]:
"""
예약(Appointments) 데이터 정제 로직
- 재전송 중복 행 제거 (Excluded)
- patient_id 빈 값 결측치 격리 (Quarantine)
- scheduled_at < booked_at 시간 역전 이상치 격리 (Quarantine)
- 상태 및 채널 코드 표준화
"""
def clean_appointments(raw_df):
    df = raw_df.copy()
    init_cnt = len(df)
    df = df.drop_duplicates(subset=["appointment_id"])
    dup_cnt = init_cnt - len(df)
    
    missing_patient = df["patient_id"].isna() | (df["patient_id"].astype(str).str.strip() == "")
    time_reversal = pd.to_datetime(df["scheduled_at"]) < pd.to_datetime(df["booked_at"])
    quarantine_mask = missing_patient | time_reversal
    
    clean_df = df[~quarantine_mask]
    quarantine_df = df[quarantine_mask]
    return clean_df, quarantine_df, dup_cnt

print("clean_appointments function defined.")